# Ensemble Learning for Trading

This notebook demonstrates how to use ensemble learning to combine multiple models for improved trading predictions.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.models.ensemble_learner import (
    BaggingEnsemble,
    BoostingEnsemble,
    StackingEnsemble,
    VotingEnsemble,
    create_ensemble
)
from src.models.base_model import BaseModel
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure ensemble learning settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Ensemble configurations
ensemble_configs = {
    'bagging': {
        'num_models': 5,
        'bootstrap_ratio': 0.8
    },
    'boosting': {
        'num_models': 5,
        'learning_rate': 0.1
    },
    'stacking': {
        'num_models': 5
    }
}

## Data Preparation

Fetch historical data and prepare it for ensemble learning.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Initialize feature engineer
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
await engineer.generate_features()

## Data Loading

Create data loaders for training and validation.

In [ ]:
def prepare_data(data, window_size=100):
    """Prepare data for training."""
    # Calculate returns
    returns = data['close'].pct_change()
    
    # Create features and targets
    features = data['features'].values
    targets = returns.shift(-1).values
    
    # Remove NaN values
    valid_idx = ~np.isnan(targets)
    features = features[valid_idx]
    targets = targets[valid_idx]
    
    # Split data
    split_idx = int(0.8 * len(features))
    
    train_features = torch.FloatTensor(features[:split_idx])
    train_targets = torch.FloatTensor(targets[:split_idx])
    val_features = torch.FloatTensor(features[split_idx:])
    val_targets = torch.FloatTensor(targets[split_idx:])
    
    return (
        train_features, train_targets,
        val_features, val_targets
    )

# Prepare data
train_features, train_targets, val_features, val_targets = prepare_data(data)

# Create datasets
train_dataset = torch.utils.data.TensorDataset(train_features, train_targets)
val_dataset = torch.utils.data.TensorDataset(val_features, val_targets)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=64
)

## Bagging Ensemble

Train and evaluate a bagging ensemble.

In [ ]:
# Initialize bagging ensemble
bagging_ensemble = BaggingEnsemble(
    base_model_class=BaseModel,
    config=config,
    **ensemble_configs['bagging']
)

# Train ensemble
bagging_history = []
num_epochs = 50

for epoch in range(num_epochs):
    epoch_metrics = []
    
    for batch in train_loader:
        features, targets = batch
        metrics = bagging_ensemble.update(features, targets)
        epoch_metrics.append(metrics)
    
    # Average metrics
    avg_metrics = {k: np.mean([m[k] for m in epoch_metrics])
                  for k in epoch_metrics[0].keys()}
    bagging_history.append(avg_metrics)
    
    print(f"Epoch {epoch+1}: {avg_metrics}")

## Boosting Ensemble

Train and evaluate a boosting ensemble.

In [ ]:
# Initialize boosting ensemble
boosting_ensemble = BoostingEnsemble(
    base_model_class=BaseModel,
    config=config,
    **ensemble_configs['boosting']
)

# Train ensemble
boosting_history = []

for epoch in range(num_epochs):
    epoch_metrics = []
    
    for batch in train_loader:
        features, targets = batch
        metrics = boosting_ensemble.update(features, targets)
        epoch_metrics.append(metrics)
    
    avg_metrics = {k: np.mean([m[k] for m in epoch_metrics])
                  for k in epoch_metrics[0].keys()}
    boosting_history.append(avg_metrics)
    
    print(f"Epoch {epoch+1}: {avg_metrics}")

## Stacking Ensemble

Train and evaluate a stacking ensemble.

In [ ]:
# Initialize stacking ensemble
stacking_ensemble = StackingEnsemble(
    base_model_class=BaseModel,
    meta_model_class=BaseModel,
    config=config,
    **ensemble_configs['stacking']
)

# Train ensemble
stacking_history = []

for epoch in range(num_epochs):
    epoch_metrics = []
    
    for batch in train_loader:
        features, targets = batch
        metrics = stacking_ensemble.update(features, targets)
        epoch_metrics.append(metrics)
    
    avg_metrics = {k: np.mean([m[k] for m in epoch_metrics])
                  for k in epoch_metrics[0].keys()}
    stacking_history.append(avg_metrics)
    
    print(f"Epoch {epoch+1}: {avg_metrics}")

## Performance Analysis

Compare the performance of different ensemble methods.

In [ ]:
def evaluate_ensemble(ensemble, data_loader):
    """Evaluate ensemble performance."""
    predictions = []
    targets = []
    
    with torch.no_grad():
        for features, batch_targets in data_loader:
            pred = ensemble.predict(features)
            predictions.append(pred)
            targets.append(batch_targets)
    
    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    
    mse = F.mse_loss(predictions, targets).item()
    mae = F.l1_loss(predictions, targets).item()
    
    return {
        'mse': mse,
        'mae': mae
    }

# Evaluate ensembles
results = {
    'bagging': evaluate_ensemble(bagging_ensemble, val_loader),
    'boosting': evaluate_ensemble(boosting_ensemble, val_loader),
    'stacking': evaluate_ensemble(stacking_ensemble, val_loader)
}

# Plot results
plt.figure(figsize=(12, 5))

# MSE comparison
plt.subplot(1, 2, 1)
plt.bar(results.keys(), [r['mse'] for r in results.values()])
plt.title('MSE Comparison')
plt.ylabel('MSE')

# MAE comparison
plt.subplot(1, 2, 2)
plt.bar(results.keys(), [r['mae'] for r in results.values()])
plt.title('MAE Comparison')
plt.ylabel('MAE')

plt.tight_layout()
plt.show()

## Training History

Analyze the training progress of each ensemble method.

In [ ]:
def plot_training_history(histories):
    """Plot training history for each ensemble method."""
    plt.figure(figsize=(15, 5))
    
    for i, (name, history) in enumerate(histories.items()):
        plt.subplot(1, 3, i+1)
        plt.plot([h['loss'] for h in history])
        plt.title(f'{name} Training Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
    
    plt.tight_layout()
    plt.show()

# Plot training history
histories = {
    'Bagging': bagging_history,
    'Boosting': boosting_history,
    'Stacking': stacking_history
}
plot_training_history(histories)

## Save Models

Save the trained ensemble models for future use.

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# Save ensembles
torch.save(bagging_ensemble.state_dict(), 'models/bagging_ensemble.pt')
torch.save(boosting_ensemble.state_dict(), 'models/boosting_ensemble.pt')
torch.save(stacking_ensemble.state_dict(), 'models/stacking_ensemble.pt')

print("Models saved successfully")